# Group-conditional conformal selection

Pooled conformal p-values compare each test point with the whole calibration set. When groups such as sites or sensors have different normal behavior, and the test batch has a different group mix than the reference data, pooled p-values are no longer valid within each group.

Passing `groups=` makes the p-values group-conditional (Mondrian): each test point is ranked only among calibration scores from its own group.

In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest

from nonconform import ConformalDetector, Split
from nonconform.metrics import false_discovery_rate, statistical_power

## Two sites with different noise levels

Site `b` is a noisier sensor: its normal readings vary twice as much. It makes up 25% of the reference data but half of the test batch. Each site also has 10 anomalies. The labels `y_test` are used only to evaluate the decisions.

In [2]:
rng = np.random.default_rng(42)
site_reference = rng.choice(np.array(["a", "b"]), size=3_000, p=[0.75, 0.25])
noise = np.where(site_reference == "b", 2.0, 1.0)[:, None]
x_reference = rng.normal(size=(3_000, 4)) * noise

# 100 normal readings per site, then 10 anomalies per site.
site_test = np.repeat(np.array(["a", "b", "a", "b"]), [100, 100, 10, 10])
noise = np.where(site_test == "b", 2.0, 1.0)[:, None]
x_test = rng.normal(size=(220, 4)) * noise
x_test[200:210] += 3.0
x_test[210:] += 6.0
y_test = np.r_[np.zeros(200, dtype=int), np.ones(20, dtype=int)]

## Pooled versus group-conditional calibration

The detector and the data split are identical. The only difference is whether the site labels are passed to `fit()` and `select()`.

In [3]:
results = {}
detectors = {}
for name, groups in [("pooled", None), ("group-conditional", site_reference)]:
    detector = ConformalDetector(
        detector=IsolationForest(random_state=42),
        strategy=Split(n_calib=0.5),
        seed=42,
    ).fit(x_reference, groups=groups)
    test_groups = None if groups is None else site_test
    selected = np.asarray(detector.select(x_test, alpha=0.1, groups=test_groups))
    detectors[name] = detector
    results[name] = {
        "discoveries": int(selected.sum()),
        "false_site_b": int(selected[100:200].sum()),
        "realized_fdp": round(float(false_discovery_rate(y_test, selected)), 3),
        "power": float(statistical_power(y_test, selected)),
    }
    print(name, results[name])

pooled {'discoveries': 21, 'false_site_b': 5, 'realized_fdp': 0.238, 'power': 0.8}
group-conditional {'discoveries': 19, 'false_site_b': 0, 'realized_fdp': 0.0, 'power': 0.95}


With pooled calibration, normal site-`b` readings are compared mostly with the quieter site `a`, so several of them become false discoveries. The noisy site-`b` calibration scores also make true site-`a` anomalies look less extreme. Group-conditional calibration avoids both effects.

The realized FDP describes this one batch. In repeated draws of the same design, the pooled procedure exceeds the 10% FDR target, while the group-conditional procedure stays below it.

## Calibration size per group

Each group's p-values cannot fall below `1 / (n_g + 1)`, where `n_g` is the number of calibration scores in that group. Check the group sizes before relying on small groups.

In [4]:
labels, counts = np.unique_counts(detectors["group-conditional"].calibration_groups)
for label, count in zip(labels, counts, strict=True):
    print(
        f"site {label}: {count} calibration scores, p-value floor {1 / (count + 1):.4f}"
    )

site a: 1135 calibration scores, p-value floor 0.0009
site b: 365 calibration scores, p-value floor 0.0027


The guarantee requires exchangeability within each site, not a matching site mix. `select()` applies BH once to the whole batch of group-conditional p-values. Group-conditional calibration supports `Split`, detached `Split`, `CrossValidation`, and `JackknifeBootstrap` with classical `Empirical` p-values in unweighted mode. See the conformal inference guide for the validity statement and full scope.